# langchain 基础
LangChain是智能体开发平台，包含一套各种帮助开发、测试、评估智能体的框架。核心包括：
- LangChain：用于快速构建智能体，可兼容任何模型提供商。
- LangGraph：从底层一步步控制智能体的构建，包括记忆（Memory）、人机协同（HITL）等
- Deep Agents：用于构建复杂的、处理多步骤的任务的智能体
- LangSmith：用于测试、观察、评估、部署智能体
这一章覆盖LangChain三个核心原语：模型、消息、工具。

参考资料
- 视频教程：https://www.bilibili.com/video/BV178w1z7EHQ?p=14
- 飞书笔记：https://my.feishu.cn/wiki/RueEwEMrpiTd6ykN3UecI0MLnrf#

## 1. 模型

`init_chat_model` 根据模型名字自动选择 provider：

对于 LangChain 支持的模型提供者（OpenAI、DeepSeek、Google、Anthropic 等），只需指定模型名称即可，它会自动确定 provider、base_url，并从环境变量读取 api_key。

In [15]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

# 调用init_chat_model函数初始化模型
model = init_chat_model(model="deepseek-chat")

print(type(model)) # <class 'langchain_deepseek.chat_models.ChatDeepSeek'>

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


对于不支持的模型，需要手动指定参数：
- 在 `.env` 中定义 `api_key` 和 `base_url`
- 在 `init_chat_model` 中指定 `model`、`model_provider="openai"`、`base_url` 和 `api_key`

In [16]:
# 对于不支持的模型，需要手动指定参数
import os

# 使用 AIHUBMIX（已在 .env 中配置）
model_aihubmix = init_chat_model(
    "qwen3.6-plus",
    model_provider="openai",
    base_url=os.getenv("AIHUBMIX_API_URL"),
    api_key=os.getenv("AIHUBMIX_API_KEY")
)
print(type(model_aihubmix))

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 2. 消息

在 LangChain 中，我们并不需要自己创建 BaseMessage 对象，LangChain 已经把常见消息根据角色（Role）创建了对应的 BaseMessage 的子类：
- **SystemMessage**：role 是 system，代表系统消息，用于设定模型角色和交互背景
- **HumanMessage**：role 是 user，代表用户输入的消息
- **AIMessage**：role 是 assistant，代表 LLM 生成的响应，包含：文本、工具调用、元数据
- **ToolMessage**：role 是 tool，代表工具调用时产生的结果

四种消息类型通过 `tool_call_id` 关联工具调用和结果：

```
SystemMessage → HumanMessage → AIMessage(tool_calls) → ToolMessage → AIMessage(最终回复)
```

下面用实际例子演示消息的使用：

In [17]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

# 四种消息类型
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Add 3 and 4.")
]

# 调用模型
response = model.invoke(messages)
print(response)

content='3 + 4 = 7' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 17, 'total_tokens': 24, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 17}, 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'e57a1c9f-86e2-4db1-a0eb-8d05a8b40f5a', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--019e2542-46aa-7c90-ac9c-d1fd2542532d-0' usage_metadata={'input_tokens': 17, 'output_tokens': 7, 'total_tokens': 24, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


可以看到，模型返回的是 `AIMessage` 对象，其中 `content` 是文本回复。如果模型决定调用工具，会在 `tool_calls` 字段中返回工具调用信息。

## 3. 工具

`@tool` 从函数签名自动生成 schema。docstring → description，类型注解 → 参数类型。

定义工具后，使用 `bind_tools` 将工具绑定到模型，模型会自动决定是否调用工具：

In [18]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`."""
    return a + b

@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`."""
    return a / b

tools = [add, multiply, divide]
tools_by_name = {t.name: t for t in tools}
model_with_tools = model.bind_tools(tools)

模型返回的 `tool_calls` 包含工具名称、参数和调用 ID。Agent 的核心就是解析这些调用，执行工具，然后把结果作为 `ToolMessage` 再发给模型。

In [19]:
# 模型自动决定是否调用工具
response = model_with_tools.invoke([HumanMessage(content="Add 3 and 4.")])
print(f"content: {response.content}")
print(f"tool_calls: {response.tool_calls}")

content: 
tool_calls: [{'name': 'add', 'args': {'a': 3, 'b': 4}, 'id': 'call_00_ywzMQsNEHulwThXjTlcn8403', 'type': 'tool_call'}]
